# ViT WeightTracker vs Physical Pruning

This notebook uses one reusable function:

1. clone the model twice
2. prune one clone physically with Torch-Pruning `BasePruner`
3. fake-prune the other clone by zeroing the same pruning groups
4. count the physical model with `fvcore.nn.FlopCountAnalysis`
5. count the zeroed model with WeightTracker active MAC calculations, not BOPs
6. print one table comparing axes and MACs per layer

In [1]:
import os
import sys
from pathlib import Path

import timm
import torch

ROOT = Path("/Users/mikkeldahl/codeq_research/CoDeQ")
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sanity_checks.pruning_weighttracker_compare import compare_weighttracker_to_physical_pruning

torch.manual_seed(0)


/Users/mikkeldahl/codeq_research/CoDeQ/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def make_vit():
    return timm.models.vision_transformer.VisionTransformer(
        img_size=32,
        patch_size=16,
        embed_dim=128,
        depth=2,
        num_heads=4,
        mlp_ratio=2,
        num_classes=10,
    ).eval()


model = make_vit()
example_inputs = torch.randn(1, 3, 32, 32)

print(model(example_inputs).shape)


torch.Size([1, 10])


In [3]:
result = compare_weighttracker_to_physical_pruning(
    model,
    example_inputs,
    pruning_ratio=0.5,
    round_to=1,
    tracker_kwargs={"prune_dim": True},
    pruner_kwargs={
        "prune_num_heads": False,
        "prune_head_dims": True,
    },
)


TypeError: iteration over a 0-d tensor

`result` contains `zeroed_model`, `physical_model`, `tracker`, `physical_analysis`, `comparison_rows`, and `summary` for manual inspection.